In [ ]:
# Cell 1: Install arc-agi from competition wheels (offline, works in Phase A and B)
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')

In [ ]:
# Cell 2: Write VericodingAgent to /tmp/my_agent.py
agent_code = '''import random, collections
import numpy as np
from agents.agent import Agent
from arcengine import GameAction


class VericodingAgent(Agent):
    """Graph exploration agent with D4 Zobrist hash, GameAction-safe."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._graph = collections.defaultdict(dict)
        self._visited = set()
        self._last_hash = None
        self._last_key = None
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)

    def _hash(self, grid):
        """D4 Zobrist: one-hot encoding, min over 8 dihedral transforms."""
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                gh, gw = g.shape[:2]
                onehot = np.eye(16, dtype=np.uint64)[np.clip(g, 0, 15)]
                hv = int(np.bitwise_xor.reduce(self._zob[:gh, :gw, :] * onehot))
                if hv < int(best): best = np.uint64(hv)
        return int(best)

    def _get_action_idx(self, action):
        if isinstance(action, int): return action
        if hasattr(action, 'action_type'): return action.action_type
        return int(action)

    def _make_action(self, idx, grid, frame):
        a = GameAction(int(idx))
        if hasattr(a, 'is_complex') and a.is_complex():
            y = random.randint(0, max(0, grid.shape[0] - 1))
            x = random.randint(0, max(0, grid.shape[1] - 1))
            a.set_data({'x': int(x), 'y': int(y)})
            return a, (int(idx), int(x), int(y))
        return a, int(idx)

    def choose_action(self, frames, latest_frame) -> GameAction:
        try: return self._fsm(frames, latest_frame)
        except Exception as e: return self._fallback(latest_frame)

    def _fsm(self, frames, latest_frame) -> GameAction:
        self._step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        raw = latest_frame.available_actions if latest_frame.available_actions else []
        if not raw: return GameAction(1)
        h, w = grid.shape[:2]

        cur_hash = self._hash(grid)
        self._visited.add(cur_hash)

        if self._last_hash is not None and self._last_key is not None:
            self._graph[self._last_hash][self._last_key] = cur_hash
        self._last_hash = cur_hash

        # Frontier: try actions that lead to unvisited states
        if cur_hash in self._graph:
            for act_key, nh in self._graph[cur_hash].items():
                if nh not in self._visited:
                    a, k = self._make_action(act_key[0] if isinstance(act_key, tuple) else act_key, grid, latest_frame)
                    self._last_key = k
                    return a
            tried = set(self._graph[cur_hash].keys())
            for a in raw:
                k = self._get_action_idx(a)
                if k not in tried:
                    _a, _k = self._make_action(k, grid, latest_frame)
                    self._last_key = _k
                    return _a

        # First visit: click non-zero cells
        complex_a, simple_a = [], []
        for a in raw:
            idx = self._get_action_idx(a)
            if idx in (6, 7, 8) or (hasattr(a, 'is_complex') and a.is_complex()):
                complex_a.append(idx)
            else:
                simple_a.append(idx)

        nz = np.count_nonzero(grid)
        if complex_a and nz > 0:
            flat = grid.flatten()
            nzi = np.where(flat > 0)[0]
            pi = nzi[self._step % len(nzi)]
            y, x = divmod(int(pi), w)
            a = GameAction(complex_a[0])
            a.set_data({'x': int(x), 'y': int(y)})
            self._last_key = (complex_a[0], int(x), int(y))
            return a

        if simple_a:
            ci = int(simple_a[self._step % len(simple_a)])
            a, k = self._make_action(ci, grid, latest_frame)
            self._last_key = k
            return a

        return self._fallback(latest_frame)

    def _fallback(self, frame) -> GameAction:
        raw = getattr(frame, 'available_actions', None) or []
        if not raw: return GameAction(1)
        c = raw[self._step % len(raw)]
        a, _ = self._make_action(self._get_action_idx(c), getattr(frame, 'frame', [[0]]), frame)
        return a
'''

with open("/tmp/my_agent.py", "w") as f:
    f.write(agent_code)
print('[Cell2] V25.1 agent written')


In [ ]:
# Cell 3: Phase B — Competition Rerun (gateway + framework)
import os, subprocess, shutil, sys

if os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    print("[Cell3] Phase B: competition rerun detected")

    # Wait for gateway sidecar
    subprocess.run([
        "curl", "--fail", "--retry", "999",
        "--retry-delay", "1", "--max-time", "3",
        "http://gateway:8001/api/games"
    ])

    # Copy framework boilerplate to working dir
    src = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/agents"
    if os.path.isdir(src):
        shutil.copytree(src, "/kaggle/working/agents", dirs_exist_ok=True)
        print("[Cell3] framework copied")
    else:
        print(f"[Cell3] WARNING: {src} not found")

    # Read our agent code written by Cell 2, write into framework location
    with open("/tmp/my_agent.py") as f:
        agent_code = f.read()
    with open("/kaggle/working/agents/my_agent.py", "w") as f:
        f.write(agent_code)

    # Write __init__.py with proper agent registration
    init = """from agents.my_agent import VericodingAgent

AVAILABLE_AGENTS = {
    "vericoding": VericodingAgent,
}
"""
    with open("/kaggle/working/agents/__init__.py", "w") as f:
        f.write(init)
    print("[Cell3] agent registered")

    # Run framework main.py
    os.chdir("/kaggle/working")
    subprocess.run([
        sys.executable, "main.py",
        "--agent", "vericoding",
        "--output", "/kaggle/working/submission.parquet",
    ])
    print("[Cell3] Phase B complete")
else:
    print("[Cell3] Phase A: skipping (no competition rerun)")


In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by framework gateway')